In [1]:
from collections import Counter

import regex

GPT4_SPLIT_PATTERN = (
    r"'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}|"
    r" ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"
)

def split_bytes(text):
    return [bytes([value]) for value in text.encode("utf-8")]

def merge_pair(parts, pair):
    merged = []
    index = 0
    while index < len(parts):
        if index + 1 < len(parts) and (parts[index], parts[index + 1]) == pair:
            merged.append(pair[0] + pair[1])
            index += 2
        else:
            merged.append(parts[index])
            index += 1
    return merged

In [2]:
class BasicTokenizer:
    def __init__(self):
        self._set_vocab({bytes([value]): value for value in range(256)})

    def _set_vocab(self, ranks, special_tokens=None):
        self.ranks = ranks
        self.special_tokens = dict(special_tokens or {})
        self.vocab = {rank: piece for piece, rank in ranks.items()}
        self.vocab.update(
            {token_id: marker.encode("utf-8") for marker, token_id in self.special_tokens.items()}
        )

    def _chunks(self, text):
        return [text]

    def train(self, text, vocab_size):
        if vocab_size < 256:
            raise ValueError("Потрібно щонайменше 256 токенів для всіх байтів UTF-8.")

        ranks = {bytes([value]): value for value in range(256)}
        chunks = [split_bytes(chunk) for chunk in self._chunks(text)]
        for token_id in range(256, vocab_size):
            counts = Counter(
                (left, right)
                for chunk in chunks
                for left, right in zip(chunk, chunk[1:])
                if left + right not in ranks
            )
            if not counts:
                break
            pair, _ = counts.most_common(1)[0]
            ranks[pair[0] + pair[1]] = token_id
            chunks = [merge_pair(chunk, pair) for chunk in chunks]

        self._set_vocab(ranks)

    def _encode_chunk(self, chunk):
        parts = split_bytes(chunk)
        while len(parts) > 1:
            ranked_pairs = (
                (self.ranks[left + right], (left, right))
                for left, right in zip(parts, parts[1:])
                if left + right in self.ranks
            )
            best = min(ranked_pairs, key=lambda entry: entry[0], default=None)
            if best is None:
                break
            parts = merge_pair(parts, best[1])
        return [self.ranks[part] for part in parts]

    def encode(self, text):
        return [token_id for chunk in self._chunks(text) for token_id in self._encode_chunk(chunk)]

    def decode(self, ids):
        return b"".join(self.vocab[token_id] for token_id in ids).decode("utf-8", errors="replace")

In [3]:
class RegexTokenizer(BasicTokenizer):
    def __init__(self, pattern=GPT4_SPLIT_PATTERN):
        super().__init__()
        self.pattern = regex.compile(pattern)

    def _chunks(self, text):
        chunks = [match.group() for match in self.pattern.finditer(text)]
        if "".join(chunks) != text:
            raise ValueError("Шаблон має охоплювати весь вхідний текст.")
        return chunks

    def encode(self, text, allowed_special=()):
        if not self.special_tokens:
            return super().encode(text)

        allowed = self.special_tokens.keys() if allowed_special == "all" else set(allowed_special)
        markers = sorted(self.special_tokens, key=len, reverse=True)
        pattern = "(" + "|".join(regex.escape(marker) for marker in markers) + ")"
        ids = []
        for index, part in enumerate(regex.split(pattern, text)):
            if index % 2 == 0:
                ids.extend(super().encode(part))
            elif part in allowed:
                ids.append(self.special_tokens[part])
            else:
                raise ValueError(f"Спеціальний токен {part!r} потрібно явно дозволити.")
        return ids

In [4]:
training_text = (
    "Привіт, світе! Модель читає текст. Hello, world! "
    "Дані 12345. Токени працюють із UTF-8. 🙂\n"
) * 20

basic = BasicTokenizer()
with_regex = RegexTokenizer()
for tokenizer in (basic, with_regex):
    tokenizer.train(training_text, vocab_size=300)

sample = "Привіт, світе! 12345 🙂"
print("Фрагменти після regex:", with_regex._chunks(sample))
print("Розмір у байтах UTF-8:", len(sample.encode("utf-8")))

for tokenizer in (basic, with_regex):
    ids = tokenizer.encode(sample)
    print(f"{type(tokenizer).__name__}: {len(ids)} токенів")
    print("ID:", ids)
    print("Байти токенів:", [tokenizer.vocab[token_id] for token_id in ids])
    print("Приклади навчених токенів:", list(tokenizer.vocab.items())[256:264])
    for example in (sample, "", "  \t\n", "Незнайомий текст: café, 日本語, 🧪"):
        assert tokenizer.decode(tokenizer.encode(example)) == example

boundary_basic = BasicTokenizer()
boundary_regex = RegexTokenizer()
for tokenizer in (boundary_basic, boundary_regex):
    tokenizer.train("a1" * 20, vocab_size=257)
assert b"a1" in boundary_basic.ranks
assert b"a1" not in boundary_regex.ranks

Фрагменти після regex: ['Привіт', ',', ' світе', '!', ' ', '123', '45', ' 🙂']
Розмір у байтах UTF-8: 36
BasicTokenizer: 13 токенів
ID: [284, 33, 32, 49, 50, 51, 52, 53, 32, 240, 159, 153, 130]
Байти токенів: [b'\xd0\x9f\xd1\x80\xd0\xb8\xd0\xb2\xd1\x96\xd1\x82, \xd1\x81\xd0\xb2\xd1\x96\xd1\x82\xd0\xb5', b'!', b' ', b'1', b'2', b'3', b'4', b'5', b' ', b'\xf0', b'\x9f', b'\x99', b'\x82']
Приклади навчених токенів: [(256, b'\xd1\x82'), (257, b'\xd1\x96'), (258, b'\xd0\xb5'), (259, b' \xd0'), (260, b'\xd0\xb8'), (261, b'\xd0\xb5\xd0'), (262, b'\xd0\xb0'), (263, b'\xd1\x80')]
RegexTokenizer: 15 токенів
ID: [276, 44, 279, 33, 32, 49, 50, 51, 52, 53, 32, 240, 159, 153, 130]
Байти токенів: [b'\xd0\x9f\xd1\x80\xd0\xb8\xd0\xb2\xd1\x96\xd1\x82', b',', b' \xd1\x81\xd0\xb2\xd1\x96\xd1\x82\xd0\xb5', b'!', b' ', b'1', b'2', b'3', b'4', b'5', b' ', b'\xf0', b'\x9f', b'\x99', b'\x82']
Приклади навчених токенів: [(256, b'\xd1\x82'), (257, b'\xd1\x96'), (258, b'\xd0\xb5'), (259, b' \xd0'), (260, b'\xd0\xb

In [5]:
import tiktoken

reference = tiktoken.get_encoding("cl100k_base")
gpt4 = RegexTokenizer(pattern=reference._pat_str)
gpt4._set_vocab(reference._mergeable_ranks, reference._special_tokens)

examples = [
    "hello world!!!? (안녕하세요!) lol123 😉",
    "Привіт, світе! 12345 🙂",
    "I'm testing\twhitespace.\r\n  café 日本語",
    "",
]
for example in examples:
    ids = gpt4.encode(example)
    assert ids == reference.encode(example)
    assert gpt4.decode(ids) == reference.decode(ids) == example
    print(repr(example), ids)

special_example = "<|endoftext|>hello world"
special_ids = gpt4.encode(special_example, allowed_special={"<|endoftext|>"})
assert special_ids == reference.encode(special_example, allowed_special="all")
assert gpt4.decode(special_ids) == special_example
try:
    gpt4.encode(special_example)
except ValueError:
    pass
else:
    raise AssertionError("Спеціальний токен має вимагати явного дозволу.")

print("Спеціальний токен:", special_ids)
print("Усі перевірки пройдено.")

'hello world!!!? (안녕하세요!) lol123 😉' [15339, 1917, 12340, 30, 320, 31495, 230, 75265, 243, 92245, 16715, 28509, 4513, 57037]
'Привіт, світе! 12345 🙂' [54745, 28089, 27385, 1830, 11, 45916, 27385, 51627, 0, 220, 4513, 1774, 28584]
"I'm testing\twhitespace.\r\n  café 日本語" [40, 2846, 7649, 197, 1336, 37174, 3304, 220, 53050, 76502, 22656, 45918, 252]
'' []
Спеціальний токен: [100257, 15339, 1917]
Усі перевірки пройдено.
